# 01. Exploratory Data Analysis & Target/Feature Audit
**Project**: CardioView 3D — Multimodal AI Hackathon 2026, Track A  
**Dataset**: Extension of Z-Alizadeh Sani Dataset (`data/extention of Z-Alizadeh sani dataset.xlsx`)  
**Hard Constraints from PROJECT_BRIEF.md**:
- NEVER use `LAD`, `LCX`, `RCA`, or `Cath` as input features (target leakage).
- All numbers and metrics must come directly from running code.
- Report only what the data shows; mark anything unclear as `[uncertain]`.


## 1. Dataset Loading, Columns, Data Types, and Missing Values
We load the dataset from `data/extention of Z-Alizadeh sani dataset.xlsx` (sheet: `Sheet 1 - Table 1`) and inspect all 59 columns.


In [1]:
import pandas as pd
import numpy as np
import os

DATA_PATH = 'data/extention of Z-Alizadeh sani dataset.xlsx' if os.path.exists('data/extention of Z-Alizadeh sani dataset.xlsx') else '../data/extention of Z-Alizadeh sani dataset.xlsx'
df = pd.read_excel(DATA_PATH, sheet_name='Sheet 1 - Table 1')

print(f"Dataset Shape: {df.shape[0]} rows (patients) x {df.shape[1]} columns")

col_summary = []
for idx, col in enumerate(df.columns, 1):
    n_missing = int(df[col].isnull().sum())
    dtype_str = str(df[col].dtype)
    n_unique = int(df[col].nunique())
    sample_val = str(df[col].iloc[0])
    col_summary.append({
        "Index": idx,
        "Column Name": col,
        "Dtype": dtype_str,
        "Missing Count": n_missing,
        "Unique Values": n_unique,
        "Sample (Row 0)": sample_val
    })

summary_df = pd.DataFrame(col_summary)
pd.set_option('display.max_rows', 65)
print(summary_df.to_string(index=False))


Dataset Shape: 303 rows (patients) x 59 columns
 Index           Column Name   Dtype  Missing Count  Unique Values     Sample (Row 0)
     1                   Age   int64              0             46                 53
     2                Weight   int64              0             54                 90
     3                Length   int64              0             44                175
     4                   Sex     str              0              2               Male
     5                   BMI float64              0            263 29.387755102040817
     6                    DM   int64              0              2                  0
     7                   HTN   int64              0              2                  1
     8        Current Smoker   int64              0              2                  1
     9             EX-Smoker   int64              0              2                  0
    10                    FH   int64              0              2                  0
    11

## 2. Target Columns Identification & Overall CAD Confirmation
The candidate target columns are:
- `Cath`: Coronary Angiography (Catheterization) overall diagnostic outcome (`CAD` vs `Normal`).
- `LAD`: Left Anterior Descending artery stenosis (`Stenotic` vs `Normal`).
- `LCX`: Left Circumflex artery stenosis (`Stenotic` vs `Normal`).
- `RCA`: Right Coronary Artery stenosis (`Stenotic` vs `Normal`).

Below we confirm from the data how `Cath` relates to the 3 vessel stenoses (`LAD`, `LCX`, `RCA`).


In [2]:
# Cross-tabulation of Cath vs (LAD, LCX, RCA)
print("=== CATH VS ARTERY STENOSIS COMBINATIONS ===")
crosstab_targets = pd.crosstab(
    df['Cath'],
    [df['LAD'], df['LCX'], df['RCA']],
    margins=True
)
print(crosstab_targets)

# Check patients where all 3 vessels are Normal
all_normal = df[(df['LAD'] == 'Normal') & (df['LCX'] == 'Normal') & (df['RCA'] == 'Normal')]
print(f"\nPatients with Normal LAD, LCX, RCA: {len(all_normal)}")
print("Cath status for these patients:")
print(all_normal['Cath'].value_counts())

# Check patients where at least one vessel is Stenotic
any_stenotic = df[(df['LAD'] == 'Stenotic') | (df['LCX'] == 'Stenotic') | (df['RCA'] == 'Stenotic')]
print(f"\nPatients with >= 1 Stenotic vessel: {len(any_stenotic)}")
print("Cath status for these patients:")
print(any_stenotic['Cath'].value_counts())

# Anomaly check
anomalies = df[((df['Cath'] == 'Normal') & ((df['LAD'] == 'Stenotic') | (df['LCX'] == 'Stenotic') | (df['RCA'] == 'Stenotic'))) |
               ((df['Cath'] == 'CAD') & ((df['LAD'] == 'Normal') & (df['LCX'] == 'Normal') & (df['RCA'] == 'Normal')))]
print(f"\nDiscrepancy count between Cath and vessel stenosis: {len(anomalies)}")
if len(anomalies) > 0:
    print(anomalies[['Age', 'Sex', 'Cath', 'LAD', 'LCX', 'RCA', 'EF-TTE', 'Typical Chest Pain']])


=== CATH VS ARTERY STENOSIS COMBINATIONS ===


LAD    Normal                            Stenotic                             \
LCX    Normal          Stenotic            Normal          Stenotic            
RCA    Normal Stenotic   Normal Stenotic   Normal Stenotic   Normal Stenotic   
Cath                                                                           
CAD         0       17       13       10       56       24       33       63   
Normal     86        0        0        0        1        0        0        0   
All        86       17       13       10       57       24       33       63   

LAD     All  
LCX          
RCA          
Cath         
CAD     216  
Normal   87  
All     303  

Patients with Normal LAD, LCX, RCA: 86
Cath status for these patients:
Cath
Normal    86
Name: count, dtype: int64

Patients with >= 1 Stenotic vessel: 217
Cath status for these patients:
Cath
CAD       216
Normal      1
Name: count, dtype: int64

Discrepancy count between Cath and vessel stenosis: 1
    Age   Sex    Cath       LAD     LC

## 3. Class Balance for All 4 Targets
We compute exact counts and percentages for each target class.


In [3]:
target_cols = ['Cath', 'LAD', 'LCX', 'RCA']
balance_records = []

for t in target_cols:
    vc = df[t].value_counts()
    for label, count in vc.items():
        pct = (count / len(df)) * 100
        balance_records.append({
            "Target": t,
            "Class Label": label,
            "Count": count,
            "Percentage (%)": round(pct, 2)
        })

balance_df = pd.DataFrame(balance_records)
print(balance_df.to_string(index=False))


Target Class Label  Count  Percentage (%)
  Cath         CAD    216           71.29
  Cath      Normal     87           28.71
   LAD    Stenotic    177           58.42
   LAD      Normal    126           41.58
   LCX      Normal    184           60.73
   LCX    Stenotic    119           39.27
   RCA      Normal    189           62.38
   RCA    Stenotic    114           37.62


## 4. Target Leakage & Data Integrity Audit
We examine columns that could leak target information directly or indirectly, plus redundant/zero-variance features.

1. **Direct Leakage (Forbidden Targets as Features)**:
   - `Cath`: The primary catheterization outcome (Overall CAD).
   - `LAD`, `LCX`, `RCA`: Individual artery catheterization stenosis outcomes.
   - *Rule*: When predicting `Cath`, none of `LAD`, `LCX`, `RCA` may be used. When predicting any individual artery, neither `Cath` nor the other arteries may be used.

2. **Physiological Territory Quasi-Leakage**:
   - `Region RWMA` (Regional Wall Motion Abnormality from echocardiogram):
     - RWMA is clinically measured during pre-catheterization echocardiography.
     - However, specific scores in this dataset have near-perfect / perfect target separation:
       - `Region RWMA == 3`: 14/14 patients (100%) have `LAD == Stenotic` and `Cath == CAD`.
       - `Region RWMA == 4`: 14/14 patients (100%) have `LAD == Stenotic` and `Cath == CAD`.
     - Marked as **[uncertain / potential quasi-leakage]** pending explicit decision.

3. **Zero-Variance Column**:
   - `Exertional CP`: Exactly 303 out of 303 rows have the value `'N'`. Zero variance; carries no predictive signal.

4. **Deterministic Collinear Pairs**:
   - `BMI` vs `(Weight, Length)`: $BMI = \frac{Weight}{(Length/100)^2}$ has maximum error 0.0 across all 303 rows.
   - `Obesity` vs `BMI`: $Obesity = 'Y'$ for all $BMI \ge 25.04$, and $'N'$ for all $BMI \le 24.91$.


In [4]:
print("=== ZERO VARIANCE AUDIT ===")
zero_var = [c for c in df.columns if df[c].nunique() <= 1]
print(f"Zero variance columns: {zero_var}")
for z in zero_var:
    print(f"  {z}: unique values = {df[z].unique().tolist()}")

print("\n=== REGION RWMA TARGET CROSS-TABULATION ===")
for t in ['Cath', 'LAD', 'LCX', 'RCA']:
    print(f"--- Region RWMA vs {t} ---")
    ct = pd.crosstab(df['Region RWMA'], df[t], normalize='index') * 100
    print(ct.round(2))
    print()

print("=== BMI and OBESITY CO-DEPENDENCY ===")
calc_bmi = df['Weight'] / ((df['Length'] / 100) ** 2)
diff = (calc_bmi - df['BMI']).abs().max()
print(f"Max absolute discrepancy between calculated BMI and dataset BMI: {diff}")
print("BMI summary by Obesity category:")
print(df.groupby('Obesity')['BMI'].agg(['count', 'min', 'median', 'max']))


=== ZERO VARIANCE AUDIT ===
Zero variance columns: ['Exertional CP']
  Exertional CP: unique values = ['N']

=== REGION RWMA TARGET CROSS-TABULATION ===
--- Region RWMA vs Cath ---


Cath            CAD  Normal
Region RWMA                
0             61.75   38.25
1             88.46   11.54
2             96.88    3.12
3            100.00    0.00
4            100.00    0.00

--- Region RWMA vs LAD ---
LAD          Normal  Stenotic
Region RWMA                  
0             53.00     47.00
1             19.23     80.77
2             18.75     81.25
3              0.00    100.00
4              0.00    100.00

--- Region RWMA vs LCX ---
LCX          Normal  Stenotic
Region RWMA                  
0             64.98     35.02
1             50.00     50.00
2             50.00     50.00
3             50.00     50.00
4             50.00     50.00

--- Region RWMA vs RCA ---


RCA          Normal  Stenotic
Region RWMA                  
0             64.52     35.48
1             61.54     38.46
2             56.25     43.75
3             57.14     42.86
4             50.00     50.00

=== BMI and OBESITY CO-DEPENDENCY ===
Max absolute discrepancy between calculated BMI and dataset BMI: 0.0
BMI summary by Obesity category:
         count        min     median        max
Obesity                                        
N           92  18.115413  23.249213  24.910767
Y          211  25.000000  28.515625  40.900658


## 5. Proposed Input Feature List
Based on the audit, we construct the input feature list:
- **Excluded**:
  - `Cath`, `LAD`, `LCX`, `RCA` (Targets / Direct Leakage)
  - `Exertional CP` (Zero variance: 303 'N')
- **Flagged for Decision**:
  - `Region RWMA`: Pre-angio echo test, but values 3 and 4 have 100% LAD stenosis and CAD.
  - `Obesity`: Direct threshold of BMI ($BMI \ge 25$). Retaining continuous `BMI` is statistically preferred.


In [5]:
targets = ['Cath', 'LAD', 'LCX', 'RCA']
direct_leakage = ['Cath', 'LAD', 'LCX', 'RCA']
zero_var_cols = ['Exertional CP']

candidate_features = [c for c in df.columns if c not in direct_leakage and c not in zero_var_cols]

print(f"Total columns in dataset: {len(df.columns)}")
print(f"Excluded direct targets: {len(direct_leakage)} {direct_leakage}")
print(f"Excluded zero-variance: {len(zero_var_cols)} {zero_var_cols}")
print(f"Candidate input features: {len(candidate_features)}")

demographic = ['Age', 'Weight', 'Length', 'Sex', 'BMI', 'Obesity']
history_symptoms = [
    'DM', 'HTN', 'Current Smoker', 'EX-Smoker', 'FH', 'CRF', 'CVA',
    'Airway disease', 'Thyroid Disease', 'CHF', 'DLP', 'BP', 'PR',
    'Edema', 'Weak Peripheral Pulse', 'Lung rales', 'Systolic Murmur',
    'Diastolic Murmur', 'Typical Chest Pain', 'Dyspnea', 'Function Class',
    'Atypical', 'Nonanginal', 'LowTH Ang'
]
ecg = ['Q Wave', 'St Elevation', 'St Depression', 'Tinversion', 'LVH', 'Poor R Progression', 'BBB']
laboratory = ['FBS', 'CR', 'TG', 'LDL', 'HDL', 'BUN', 'ESR', 'HB', 'K', 'Na', 'WBC', 'Lymph', 'Neut', 'PLT']
echo = ['EF-TTE', 'Region RWMA', 'VHD']

categorized_all = demographic + history_symptoms + ecg + laboratory + echo
assert set(categorized_all) == set(candidate_features), "Mismatch in categorized features!"

feature_table = []
for cat_name, feats in [
    ('Demographic & Anthropometric', demographic),
    ('History & Clinical Symptoms', history_symptoms),
    ('ECG Findings', ecg),
    ('Laboratory Tests', laboratory),
    ('Echocardiography', echo)
]:
    for f in feats:
        dtype = str(df[f].dtype)
        n_uniq = df[f].nunique()
        flag = ''
        if f == 'Region RWMA':
            flag = '[uncertain: quasi-leakage risk / 100% LAD stenosis in classes 3 & 4]'
        elif f == 'Obesity':
            flag = '[collinear with BMI >= 25]'
        elif f in ['Weight', 'Length']:
            flag = '[collinear with BMI formula]'
        feature_table.append({
            "Domain": cat_name,
            "Feature Name": f,
            "Dtype": dtype,
            "Unique Values": n_uniq,
            "Clinical / Modeling Notes": flag if flag else "Standard clinical input"
        })

feat_df = pd.DataFrame(feature_table)
print(feat_df.to_string(index=False))


Total columns in dataset: 59
Excluded direct targets: 4 ['Cath', 'LAD', 'LCX', 'RCA']
Excluded zero-variance: 1 ['Exertional CP']
Candidate input features: 54


                      Domain          Feature Name   Dtype  Unique Values                                            Clinical / Modeling Notes
Demographic & Anthropometric                   Age   int64             46                                              Standard clinical input
Demographic & Anthropometric                Weight   int64             54                                         [collinear with BMI formula]
Demographic & Anthropometric                Length   int64             44                                         [collinear with BMI formula]
Demographic & Anthropometric                   Sex     str              2                                              Standard clinical input
Demographic & Anthropometric                   BMI float64            263                                              Standard clinical input
Demographic & Anthropometric               Obesity     str              2                                           [collinear with BMI >= 25]